## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(viridis)
library(ggrepel)

## Visualize DEG Analysis

### Read in Data

In [4]:
# List all DEG files
files <- list.files(path = "05_DEGs", pattern = "\\_results.csv$", full.names = TRUE)

# View
head(files)

[1] "05_DEGs/R270X_D18_vs_R270X_D46_results.csv"
[2] "05_DEGs/R270X_D18_vs_R270X_D90_results.csv"
[3] "05_DEGs/R270X_D18_vs_WT_D18_results.csv"   
[4] "05_DEGs/R270X_D18_vs_WT_D46_results.csv"   
[5] "05_DEGs/R270X_D18_vs_WT_D90_results.csv"   
[6] "05_DEGs/R270X_D46_vs_R270X_D90_results.csv"

### Make Volcano Plot

In [5]:
create_volcano_plot <- function(file) {

  # Format output name
  filename <- gsub("\\_results.csv$", "", basename(file))
  print(filename)
    
  # Read the CSV file
  data <- read.csv(file)
  
  # FIX: Create consistent significance definition for both coloring and filtering
  data$Significant <- ifelse(data$adj.P.Val < 0.05 & abs(data$logFC) > 1, "Yes", "No")
  
  # FIX: Use the same logic for coloring
  data$Color <- ifelse(data$adj.P.Val < 0.05 & data$logFC > 1, "Upregulated", 
                       ifelse(data$adj.P.Val < 0.05 & data$logFC < -1, "Downregulated", "Not Significant"))
  
  # Get the top 10 significant upregulated genes
  top_upregulated <- data %>%
    filter(Color == "Upregulated") %>%
    arrange(desc(logFC)) %>%
    head(10)
  
  # Get the top 10 significant downregulated
  top_downregulated <- data %>%
    filter(Color == "Downregulated") %>%
    arrange(logFC) %>%
    head(10)

  # Skip plotting if less than 5 upregulated or downregulated DEGs
  if (nrow(top_upregulated) < 5 || nrow(top_downregulated) < 5) {
    cat("Not enough data to plot", file, "\n")
    return()
  }

  # Print the number of genes in each category
  print(paste("Number of upregulated genes:", nrow(top_upregulated)))
  print(paste("Number of downregulated genes:", nrow(top_downregulated)))
    
  # Combine the top genes for labeling
  top_genes <- rbind(top_upregulated, top_downregulated)
    
  # Initialize PDF
  pdf(file = paste0("07_DEG_Figures/volcano_plot_", filename, ".pdf"), width = 8, height = 6)
  
  # Factor the Color column 
  data$Color <- factor(data$Color, levels = c("Downregulated", "Not Significant", "Upregulated"))
  
  # Calculate the limits for the axes
  x_min <- min(data$logFC, na.rm = TRUE) - 0.1 
  x_max <- max(data$logFC, na.rm = TRUE) + 0.1 
  y_min <- 0
  y_max <- max(-log10(data$adj.P.Val), na.rm = TRUE) + 0.1
   
  # Generate the volcano plot
  p <- ggplot(data, aes(x = logFC, y = -log10(adj.P.Val), color = Color)) +
    geom_point(alpha = 0.6) +
    scale_color_manual(
      values = c("Downregulated" = "blue", "Not Significant" = "grey", "Upregulated" = "red"),
      name = "Significance"
    ) +
    labs(title = paste("Volcano Plot for", filename),
         x = "Log2 Fold Change",
         y = "-Log10 Adjusted P-Value") +
    scale_x_continuous(limits = c(x_min, x_max), expand = expansion(mult = c(0.05, 0.05))) +  
    scale_y_continuous(limits = c(y_min, y_max), expand = expansion(mult = c(0.05, 0.05))) +  
    theme_minimal() +
    theme(legend.position = "top", 
          plot.title = element_text(hjust = 0.5)) + 
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "darkgrey") +
    geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "darkgrey") +
    geom_label_repel(data = top_genes, aes(label = X), 
                     size = 3, 
                     box.padding = 0.5, 
                     point.padding = 0.5, 
                     segment.color = 'grey50', 
                     show.legend = FALSE,
                     max.overlaps = Inf)
  
  # Print the plot
  print(p)
  
  # Close the PDF device
  dev.off()
}

In [6]:
# Create volcano plots
for (file in files) {
  create_volcano_plot(file)
}

[1] "R270X_D18_vs_R270X_D46"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D18_vs_R270X_D90"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D18_vs_WT_D18"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D18_vs_WT_D46"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D18_vs_WT_D90"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D46_vs_R270X_D90"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D46_vs_WT_D46"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D46_vs_WT_D90"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "R270X_D90_vs_WT_D90"
[1] "Number of upregulated genes: 10"
[1] "Number of downregulated genes: 10"
[1] "WT_D18_vs_R270X_D46"
[1] "Number of upregulated ge

## Visualize DEG GO Analysis

### Read in Data

In [7]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Panther_2016", "Reactome_2022", "RNAseq_DiseaseGene_DrugSigs_GEO")

In [8]:
# Store GO data into a data frame

# Initialize an empty dataframe
all_data <- data.frame()

# Specify the directory containing the files
directory <- "06_DEG_GO"

# List all files in the directory that match the pattern
file_list <- list.files(path = directory, pattern = "results_enrichr_results\\.xlsx$", full.names = TRUE)

# Iterate over each file and read in the corresponding Excel sheets
for (file_path in file_list) {
  # Extract the comparison name from the file name
  comparison <- tools::file_path_sans_ext(basename(file_path))
      
  # Get the names of the sheets in the Excel file
  databases <- excel_sheets(file_path)
  
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file_path, sheet = database)
      
      if (nrow(df) == 0) {
        next
      }
      
      # Select the required columns and add module (comparison) and database information
      df <- df %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = comparison, Database = database)

      # Filter out rows where the Adjusted.P.value is less than or equal to 0.05
      df <- df %>%
      filter(Adjusted.P.value <= 0.05)
        
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Rename
all_data$Module <- gsub("_results_enrichr_results$", "", all_data$Module)

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Formyltetrahydroformate biosynthesis Homo sapiens P02743,0.0445892,55.29167,R270X_D46_vs_WT_D46,Panther_2016
2,De novo pyrimidine deoxyribonucleotide biosynthesis Homo sapiens P02739,0.0445892,24.56790,R270X_D46_vs_WT_D46,Panther_2016


In [9]:
# Count the number of total terms and unique terms for each database
database_term_counts <- all_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(database_term_counts)

# A tibble: 1 × 3
  Database     Total_Terms Unique_Terms
  <chr>              <int>        <int>
1 Panther_2016           2            2


### Plot Top 25 Dot Plots

In [10]:
# Calculate the number of modules each term appears in for each database
term_module_counts <- all_data %>%
  group_by(Database, Term) %>%
  summarise(ModuleCount = n_distinct(Module), .groups = 'drop')

# Rank the terms within each database by the number of modules they appear in
ranked_terms <- term_module_counts %>%
  arrange(Database, desc(ModuleCount)) %>%
  group_by(Database) %>%
  slice_head(n = 25) %>%
  ungroup()

# Merge with the original data to filter the top 25 terms per database
filtered_data_top_25 <- all_data %>%
  semi_join(ranked_terms, by = c("Database", "Term"))

# Print the filtered data
head(filtered_data_top_25)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Formyltetrahydroformate biosynthesis Homo sapiens P02743,0.0445892,55.29167,R270X_D46_vs_WT_D46,Panther_2016
2,De novo pyrimidine deoxyribonucleotide biosynthesis Homo sapiens P02739,0.0445892,24.56790,R270X_D46_vs_WT_D46,Panther_2016


In [11]:
# Count the number of total terms and unique terms for each database
filtered_database_term_counts <- filtered_data_top_25 %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(filtered_database_term_counts)

# A tibble: 1 × 3
  Database     Total_Terms Unique_Terms
  <chr>              <int>        <int>
1 Panther_2016           2            2


In [12]:
# Create plots for each database
for (database in unique(filtered_data_top_25$Database)) {
  database_filtered_data <- filtered_data_top_25 %>%
    filter(Database == database)
  
  if (nrow(database_filtered_data) > 0) {
    dot_plot <- ggplot(database_filtered_data, aes(x = Module, y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
      geom_point(shape = 21) +
      scale_fill_viridis() +
      xlab('') + ylab('') +
      labs(
        title = 'Top GO Terms Associated with Significant DEGs',
        subtitle = glue('{database}')
      ) +
      theme(
        panel.background = element_rect(fill = "white", color = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        panel.border = element_blank(),
        axis.line = element_line(color = "black"),
        plot.background = element_rect(fill = "white", color = NA),
        axis.text.x = element_text(angle = 90, hjust = 1)
      )
    
    # Save the dot plot with the database name in the filename
    ggsave(filename = glue("07_DEG_Figures/top_25_dot_plot_{database}.pdf"), plot = dot_plot, height = 10, width = 12)
  } else {
    cat(glue("No data available for {database}. Skipping...\n"))
  }
}